In [1]:
GPT_CONFIG_124M = {
    "vocab_size": 50257, # 어휘 사전 크기
    "context_length": 1024, # 컨텍스트 길이
    "emb_dim": 124, # 모델 차원
    "n_heads": 12, # 어텐션 헤드 수
    "n_layers": 12, # 트랜스포머 블록 수
    "drop_rate": 0.1, # 드롭아웃 확률
    "qkv_bias": False, # 쿼리, 키, 값에 대한 편향 사용 여부
}

In [2]:
import torch
import torch.nn as nn

class DummyGPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )
        self.final_norm = DummyLayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False)
        
    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)  # (batch_size, seq_len, emb_dim)
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
        )
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)  # (batch_size, seq_len, vocab_size)
        return logits
    
class DummyTransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        
    def forward(self, x):
        return x  # Dummy implementation, no actual transformation
    
class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        
    def forward(self, x):
        return x  # Dummy implementation, no actual normalization

In [6]:
import tiktoken

tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every dat holds a"

batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)  # (batch_size, seq_len)

print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 4818, 6622,  257]])


In [7]:
torch.manual_seed(123)
model = DummyGPTModel(GPT_CONFIG_124M)
logits = model(batch)
print("출력의 크기: ", logits.shape)  # (batch_size, seq_len, vocab_size)
print(logits)

출력의 크기:  torch.Size([2, 4, 50257])
tensor([[[ 0.0845, -0.7562, -0.5155,  ...,  0.1464, -0.3696, -0.8465],
         [-1.8695,  0.9306,  0.5656,  ...,  1.6385, -0.4041,  0.7629],
         [-0.8128, -0.1258,  0.0963,  ...,  0.5189, -0.2025, -1.8511],
         [ 0.5516, -0.5280,  1.1438,  ...,  0.5581,  0.1561, -0.1914]],

        [[-0.0942, -0.6662, -0.8182,  ..., -0.4637, -0.8010, -1.1613],
         [-0.5418,  1.3387, -0.2776,  ...,  1.9998,  0.2507,  1.0242],
         [-0.3663,  0.0034,  0.1838,  ...,  1.0916, -0.0851, -0.9881],
         [ 0.6574,  0.8107,  0.4187,  ..., -0.6624,  0.0583, -1.5753]]],
       grad_fn=<UnsafeViewBackward0>)


#### 4.2 층 정규화(Layer Normalization)을 활성화 정규화하기  
많은 층을 가진 심층 신경망을 훈련하는 것은 vanishing gradient or exploding gradient와 같은 문제를 일으킴.  
이런 문제로 인해 훈련 과정이 불안정해지고 신경망이 가중치를 효과적으로 조정하기 어렵게 만듦. 이는 훈련 과정에서  
손실함수를 최소화 하는 신경망 파라미터 집합을 찾기 어렵다는 의미임. 즉, 신경망이 정확한 예측이나 결정을 내리기 위해서 필요한 데이터에  
패턴을 학습하기 어렵다는 것이다. 

layer normalization의 핵심 아이디어는 신경망 층의 활성화(출력)를 평균이 0이고 분산이 1이 되도록 조정하는 것임.  
이런 가중치 조정이 수렴 속도를 높이며 일관되고 안정적인 훈련을 보장함. GPT-2와 현대적인 트랜스포머 구조에서는  
층 정규화가 멀티 헤드 어텐션 모듈 이전과 이후에 적용되며, DummyLayerNorm 클래스에서 보았듯이 최종 출력 층 이전에도  
적용된다. 

In [9]:
torch.manual_seed(123)
batch_example = torch.randn(2, 5)
layer = nn.Sequential(nn.Linear(5, 6), nn.ReLU())
out = layer(batch_example)
print(out)

tensor([[0.2260, 0.3470, 0.0000, 0.2216, 0.0000, 0.0000],
        [0.2133, 0.2394, 0.0000, 0.5198, 0.3297, 0.0000]],
       grad_fn=<ReluBackward0>)


In [10]:
mean = out.mean(dim=-1, keepdim=True)
var = out.var(dim=-1, keepdim=True)
print("평균: ", mean)
print("분산: ", var)

평균:  tensor([[0.1324],
        [0.2170]], grad_fn=<MeanBackward1>)
분산:  tensor([[0.0231],
        [0.0398]], grad_fn=<VarBackward0>)


In [21]:
out_norm = (out - mean) / torch.sqrt(var)
mean = out_norm.mean(dim=-1, keepdim=True)
var = out_norm.var(dim=-1, keepdim=True)
print("정규화된 출력: ", out_norm)
print("정규화 후 평균: ", mean)
print("정규화 후 분산: ", var)

정규화된 출력:  tensor([[ 0.6160,  1.4127, -0.8717,  0.5874, -0.8717, -0.8717],
        [-0.0187,  0.1123, -1.0875,  1.5175,  0.5649, -1.0875]],
       grad_fn=<DivBackward0>)
정규화 후 평균:  tensor([[0.0001],
        [0.0002]], grad_fn=<MeanBackward1>)
정규화 후 분산:  tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [22]:
torch.set_printoptions(sci_mode=False)
print("평균: \n", mean)
print("분산: \n", var)

평균: 
 tensor([[0.0001],
        [0.0002]], grad_fn=<MeanBackward1>)
분산: 
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [23]:
class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
        self.eps = 1e-5
        
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift

이 layer normalization의 구현은 입력 텐서 x의 마지막 차원인 임베딩 차원(emb_dim)에 대해 작동함. 변수 eps는 당연히 정규화 시에  
분모가 0이 되는 것을 막기 위해 분산에 더해지는 작은 상수이고, scale과 shift는 입력과 차원이 동일한 훈련 가능한 파라미터이다.  
LLM이 훈련하는 동안 두 파라미터를 조정하는 것이 훈련 작업에서 모델의 성능을 향상시킨다고 판단하는 경우 자동으로 조정함.  
이를 통해서 모델은 처리하는 데이터에 가장 잘 맞는 스케일 조정과 이동을 학습할 수 있음.

편향된 분산  
앞의 구현에서 unbiased=False로 지정하여 분산을 계산했음. 이와 같이 설정하면 분산을 계산할 때 입력의 개수 n으로 나눔,  
즉, 이 방식은 표본 분산 추정의 편향을 조정하기 위해 n-1로 나누는 베셀 보정을 적용하지 않음. 따라서 이렇게 계산하면 편향된  
분산을 추정하게 됨. 임베딩 차원 n이 매우 큰 LLM의 경우 n과 n-1을 사용하는 차이는 무시할 수 있음. 여기에서는 GPT-2 모델의  
정규화와 호환성을 위해 이런 방식을 사용함. 텐서플로의 기본 동작 방식이 이렇기 때문에 원본 GPT-2 모델이 이렇게 구현됨.

In [27]:
ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)
mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, keepdim=True, unbiased=False)
print("평균: ", mean)
print("분산: ", var)


평균:  tensor([[    -0.0000],
        [     0.0000]], grad_fn=<MeanBackward1>)
분산:  tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


### 4.3 GELU 활성화 함수를 사용하는 피드포워드 네트워크 구현하기  
LLM의 트랜스포머 블록에서 사용되는 신경망 하위 모듈을 구현해보자!  
역사적으로 ReLU 활성화 함수가 딥러닝에서 널리 사용됨. 하지만 LLM에서는 ReLU 대신 여러 다른 활성화 함수를 사용함.  
대표적인 함수는 GELU(Gaussian Error Linear Unit)와 SwiGLU(Swish-gated linear unit) 2개임.  
$GELU(x) 0.5 * x ** (1+tanh(sqrt()$